# Anonymization & De-Anonymization with LangChain

### The full flow:
```
Original Input (real PII)
        ↓
  [ANONYMIZER]  → replaces PII with fake-but-realistic values
        ↓        → stores a mapping: fake → real
       LLM       → works on anonymized text (never sees real data)
        ↓
  [DE-ANONYMIZER] → swaps fake values back to real ones
        ↓
Final Output (real names/emails restored)
```

**Key difference from just masking:**
| Masking | Anonymization + De-anonymization |
|---|---|
| `john@gmail.com` → `[EMAIL MASKED]` | `john@gmail.com` → `fake_xyz@email.com` |
| LLM sees a blank | LLM sees a realistic fake value |
| Output has `[EMAIL MASKED]` forever | Output gets real value restored |
| One-way | Reversible ✅ |

In [1]:
# Install required libraries
!pip install langchain langchain-openai langchain-experimental
!pip install presidio-analyzer presidio-anonymizer spacy faker
!python -m spacy download en_core_web_lg

     ---------------------------------------- 0.0/400.7 MB ? eta -:--:--
     ---------------------------------------- 0.0/400.7 MB ? eta -:--:--
     ---------------------------------------- 0.0/400.7 MB ? eta -:--:--
     ---------------------------------------- 0.3/400.7 MB ? eta -:--:--
     ---------------------------------------- 0.8/400.7 MB 2.8 MB/s eta 0:02:23
     ---------------------------------------- 1.8/400.7 MB 3.6 MB/s eta 0:01:51
     ---------------------------------------- 2.6/400.7 MB 3.9 MB/s eta 0:01:43
     ---------------------------------------- 3.7/400.7 MB 4.0 MB/s eta 0:01:39
     ---------------------------------------- 4.5/400.7 MB 4.1 MB/s eta 0:01:38
      --------------------------------------- 5.5/400.7 MB 4.0 MB/s eta 0:01:38
      --------------------------------------- 6.0/400.7 MB 3.8 MB/s eta 0:01:43
      --------------------------------------- 6.6/400.7 MB 3.7 MB/s eta 0:01:46
      --------------------------------------- 7.6/400.7 MB 3.8 MB/s 

In [2]:
from dotenv import load_dotenv
import httpx

load_dotenv()

True

---
## Step 1 — Understand the Anonymizer

`PresidioReversibleAnonymizer` does two things:
1. **Detects** PII using Presidio
2. **Replaces** each PII value with a **fake but realistic** value (using Faker library)
3. **Remembers** the mapping so it can reverse it later

```
john.doe@gmail.com   →  olivia.jones@yahoo.com   (fake email — looks real)
+91-9876543210       →  +1-555-234-5678          (fake phone — looks real)
John Doe             →  Michael Smith            (fake name — looks real)
```

In [3]:
from langchain_experimental.data_anonymizer import PresidioReversibleAnonymizer

# Initialize the reversible anonymizer
# analyzed_fields = which PII types to detect and anonymize
anonymizer = PresidioReversibleAnonymizer(
    analyzed_fields=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "LOCATION"]
)

print("✅ Reversible anonymizer ready")

✅ Reversible anonymizer ready


---
## Step 2 — Anonymize the Input

In [4]:
original_text = """
Hi, I am Ravi Kumar from Mumbai. My email is ravi.kumar@techcorp.com 
and you can call me at +91-9876543210. 
Please send the contract to my colleague Priya Sharma at priya.sharma@techcorp.com.
"""

print("ORIGINAL TEXT (with real PII):")
print(original_text)
print("="*60)

# Anonymize — replaces PII with realistic fake values
anonymized_text = anonymizer.anonymize(original_text)

print("ANONYMIZED TEXT (sent to LLM):")
print(anonymized_text)

ORIGINAL TEXT (with real PII):

Hi, I am Ravi Kumar from Mumbai. My email is ravi.kumar@techcorp.com 
and you can call me at +91-9876543210. 
Please send the contract to my colleague Priya Sharma at priya.sharma@techcorp.com.

ANONYMIZED TEXT (sent to LLM):

Hi, I am Jamie Huang from Lake Tammy. My email is omckinney@example.net 
and you can call me at 425-665-4745. 
Please send the contract to my colleague Janice Ellis at jennifer16@example.net.



In [5]:
# --- Inspect the mapping that was created ---
# This is how de-anonymization works later

print("ANONYMIZATION MAPPING (fake → real):")
print("-"*60)

for fake_value, real_value in anonymizer.deanonymizer_mapping.items():
    print(f"  Fake: {fake_value:<35} → Real: {real_value}")

ANONYMIZATION MAPPING (fake → real):
------------------------------------------------------------
  Fake: PERSON                              → Real: {'Jamie Huang': 'Ravi Kumar', 'Janice Ellis': 'Priya Sharma'}
  Fake: LOCATION                            → Real: {'Lake Tammy': 'Mumbai'}
  Fake: EMAIL_ADDRESS                       → Real: {'omckinney@example.net': 'ravi.kumar@techcorp.com', 'jennifer16@example.net': 'priya.sharma@techcorp.com'}
  Fake: PHONE_NUMBER                        → Real: {'425-665-4745': '+91-9876543210'}


---
## Step 3 — Send Anonymized Text to LLM

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    http_client=httpx.Client(verify=False)
)

# LLM only ever sees the anonymized text
llm_response = llm.invoke([
    SystemMessage(content="You are a helpful assistant. Write a short formal introduction email based on the info provided."),
    HumanMessage(content=anonymized_text)
])

llm_output = llm_response.content

print("LLM OUTPUT (with fake values):")
print(llm_output)

LLM OUTPUT (with fake values):
Subject: Introduction and Contract Request

Dear [Recipient's Name],

I hope this message finds you well. My name is Jamie Huang, and I am reaching out from Lake Tammy. You can contact me at omckinney@example.net or by phone at 425-665-4745.

I would like to request that the contract be sent to my colleague, Janice Ellis, at jennifer16@example.net.

Thank you for your assistance.

Best regards,

Jamie Huang  
Lake Tammy  
omckinney@example.net  
425-665-4745  


---
## Step 4 — De-Anonymize the Output

Now swap the fake values back to the real ones in the LLM's response.

In [8]:
# De-anonymize — replace fake values with real ones in the LLM output
final_output = anonymizer.deanonymize(llm_output)

print("FINAL OUTPUT (real values restored):")
print("="*60)
print(final_output)

print("\n" + "="*60)
print("SIDE BY SIDE:")
print(f"LLM saw    → fake values like: {list(anonymizer.deanonymizer_mapping.keys())[:2]}")
print(f"User sees  → real values like: {list(anonymizer.deanonymizer_mapping.values())[:2]}")

FINAL OUTPUT (real values restored):
Subject: Introduction and Contract Request

Dear [Recipient's Name],

I hope this message finds you well. My name is Ravi Kumar, and I am reaching out from Mumbai. You can contact me at ravi.kumar@techcorp.com or by phone at +91-9876543210.

I would like to request that the contract be sent to my colleague, Priya Sharma, at priya.sharma@techcorp.com.

Thank you for your assistance.

Best regards,

Ravi Kumar  
Mumbai  
ravi.kumar@techcorp.com  
+91-9876543210  

SIDE BY SIDE:
LLM saw    → fake values like: ['PERSON', 'LOCATION']
User sees  → real values like: [{'Jamie Huang': 'Ravi Kumar', 'Janice Ellis': 'Priya Sharma'}, {'Lake Tammy': 'Mumbai'}]


---
## Step 5 — Build a Reusable Pipeline with LangChain

Now wire everything together as a proper LangChain chain using `RunnableLambda`.

In [9]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate

# Fresh anonymizer for this pipeline
pipeline_anonymizer = PresidioReversibleAnonymizer(
    analyzed_fields=["PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "LOCATION"]
)

# Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer clearly and professionally."),
    ("human", "{input}")
])

# Step functions
def anonymize_step(input_dict: dict) -> dict:
    """Anonymize input before sending to LLM"""
    original = input_dict["input"]
    anonymized = pipeline_anonymizer.anonymize(original)

    print(f"🔒 ANONYMIZED INPUT:")
    print(f"   Original  : {original.strip()}")
    print(f"   Anonymized: {anonymized.strip()}")
    print("-"*60)

    return {"input": anonymized}


def deanonymize_step(llm_output) -> str:
    """De-anonymize LLM output to restore real values"""
    raw = llm_output.content
    restored = pipeline_anonymizer.deanonymize(raw)

    print(f"🔓 DE-ANONYMIZED OUTPUT:")
    print(f"   LLM said : {raw.strip()}")
    print(f"   Restored : {restored.strip()}")
    print("-"*60)

    return restored


# Wire the full chain
anon_chain = (
    RunnableLambda(anonymize_step)    # Step 1: anonymize input
    | prompt                          # Step 2: format prompt
    | llm                             # Step 3: LLM call
    | RunnableLambda(deanonymize_step) # Step 4: restore real values
)

print("✅ Anonymization pipeline ready")

✅ Anonymization pipeline ready


In [10]:
# --- TEST 1: Introduction email ---
print("TEST 1: Introduction email with PII")
print("="*60)

result = anon_chain.invoke({
    "input": "Write a short intro for Arjun Mehta from Delhi, email arjun.mehta@infosys.com, phone +91-9123456789"
})

print(f"\n✅ FINAL ANSWER (real PII restored):")
print(result)

TEST 1: Introduction email with PII
🔒 ANONYMIZED INPUT:
   Original  : Write a short intro for Arjun Mehta from Delhi, email arjun.mehta@infosys.com, phone +91-9123456789
   Anonymized: Write a short intro for Thomas Fischer from Lewisborough, email jparks@example.net, phone +1-429-583-5413
------------------------------------------------------------
🔓 DE-ANONYMIZED OUTPUT:
   LLM said : Certainly! Here’s a short introduction for Thomas Fischer:

---

**Introduction:**

Meet Thomas Fischer, a dedicated professional based in Lewisborough. With a strong commitment to excellence, Thomas is known for his innovative approach and problem-solving skills. He can be reached via email at jparks@example.net or by phone at +1-429-583-5413. Thomas is always eager to connect and collaborate on new opportunities.

--- 

Feel free to modify any part of this introduction as needed!
   Restored : Certainly! Here’s a short introduction for Arjun Mehta:

---

**Introduction:**

Meet Arjun Mehta, a dedicat

In [11]:
# --- TEST 2: Multiple people ---
print("TEST 2: Multiple people's PII")
print("="*60)

# Reset anonymizer for fresh mapping
pipeline_anonymizer.reset_deanonymizer_mapping()

result = anon_chain.invoke({
    "input": """
    Summarize a meeting between:
    - Sneha Patel (sneha.patel@startup.io, +91-9988001122) from Pune
    - David Wilson (d.wilson@globalcorp.com, +1-415-555-0199) from San Francisco
    They discussed a new AI product partnership.
    """
})

print(f"\n✅ FINAL ANSWER (real PII restored):")
print(result)

TEST 2: Multiple people's PII
🔒 ANONYMIZED INPUT:
   Original  : Summarize a meeting between:
    - Sneha Patel (sneha.patel@startup.io, +91-9988001122) from Pune
    - David Wilson (d.wilson@globalcorp.com, +1-415-555-0199) from San Francisco
    They discussed a new AI product partnership.
   Anonymized: Summarize a meeting between:
    - Sneha Christine Castillo (jordanmichele@example.net, (212)418-0100x950) from Pune
    - David Wilson (doylerobert@example.com, 798.779.1066x579) from Port Susanfurt
    They discussed a new AI product partnership.
------------------------------------------------------------
🔓 DE-ANONYMIZED OUTPUT:
   LLM said : In the recent meeting between Sneha Christine Castillo from Pune and David Wilson from Port Susanfurt, the primary focus was on exploring a potential partnership for a new AI product. Both parties discussed their respective expertise and resources that could contribute to the development and success of the product. They outlined key objective

In [12]:
# --- TEST 3: No PII — should pass through unchanged ---
print("TEST 3: No PII in input")
print("="*60)

pipeline_anonymizer.reset_deanonymizer_mapping()

result = anon_chain.invoke({
    "input": "What is the capital of Japan and what is it known for?"
})

print(f"\n✅ FINAL ANSWER:")
print(result)

TEST 3: No PII in input
🔒 ANONYMIZED INPUT:
   Original  : What is the capital of Japan and what is it known for?
   Anonymized: What is the capital of West Petermouth and what is it known for?
------------------------------------------------------------
🔓 DE-ANONYMIZED OUTPUT:
   LLM said : West Petermouth is a fictional location and does not have a real-world capital or significance. If you are referring to a specific book, movie, or other fictional work, please provide more context, and I would be happy to help with information related to that.
   Restored : Japan is a fictional location and does not have a real-world capital or significance. If you are referring to a specific book, movie, or other fictional work, please provide more context, and I would be happy to help with information related to that.
------------------------------------------------------------

✅ FINAL ANSWER:
Japan is a fictional location and does not have a real-world capital or significance. If you are referr

---
## Step 6 — Inspect the Mapping at Any Point

In [13]:
# You can always inspect what got anonymized in the last run

print("LAST RUN — Anonymization Mapping:")
print("-"*60)

mapping = pipeline_anonymizer.deanonymizer_mapping

if mapping:
    for fake, real in mapping.items():
        print(f"  {fake:<40} ← restored to → {real}")
else:
    print("  (no PII was found in the last run)")

LAST RUN — Anonymization Mapping:
------------------------------------------------------------
  LOCATION                                 ← restored to → {'West Petermouth': 'Japan'}


---
## Summary

```
PresidioReversibleAnonymizer  (from langchain_experimental)
  .anonymize(text)              → detects PII, replaces with fake values, stores mapping
  .deanonymize(text)            → swaps fake values back to real ones
  .deanonymizer_mapping         → inspect the fake→real dictionary
  .reset_deanonymizer_mapping() → clear mapping for a fresh run

Pipeline:
  RunnableLambda(anonymize_step)    ← wraps anonymizer around input
  | prompt | llm                    ← LLM never sees real PII
  | RunnableLambda(deanonymize_step) ← restores real values in output
```

| Stage | What happens |
|---|---|
| Input | `Ravi Kumar, ravi@techcorp.com` |
| After anonymize | `John Smith, james.wilson@yahoo.com` |
| LLM sees | fake values only |
| LLM output | `...James Wilson at james.wilson@yahoo.com...` |
| After deanonymize | `...Ravi Kumar at ravi@techcorp.com...` |